## 02 — Adding a GeoJSON Layer

In the last lesson we added individual markers — one Python object per feature. That approach breaks down fast: loading 500 city points means 500 `Marker` calls.

**GeoJSON layers** solve this. ipyleaflet's `GeoJSON` class takes a whole FeatureCollection and renders every feature in one shot — points, lines, and polygons together.

## Coordinate Order: GeoJSON vs ipyleaflet

Before writing any code, the most important thing to know:

| Where | Order | Example |
|---|---|---|
| GeoJSON | `[longitude, latitude]` | `[-98.49, 33.91]` |
| ipyleaflet `Map`/`Marker` | `(latitude, longitude)` | `(33.91, -98.49)` |

GeoJSON follows the mathematical `(x, y)` convention — longitude is the x-axis. ipyleaflet's `Map` and `Marker` use the geographic convention — latitude first.

The `GeoJSON` layer class reads a GeoJSON dict directly, so its coordinates stay in GeoJSON order `[lon, lat]`. You do not need to swap them.

## Adding a GeoJSON Layer from a Dict

The simplest case: pass a GeoJSON dict straight to `GeoJSON()` and add it to the map.

In [1]:
from ipyleaflet import Map, GeoJSON

WICHITA_FALLS = (33.9137, -98.4934)  # (lat, lon) — ipyleaflet order

# A minimal FeatureCollection with two points
data = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [-98.4934, 33.9137]},  # [lon, lat]
            "properties": {"name": "City Hall"}
        },
        {
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [-98.5187, 33.8956]},
            "properties": {"name": "Midwestern State University"}
        }
    ]
}

m = Map(center=WICHITA_FALLS, zoom=12)
layer = GeoJSON(data=data)
m.add(layer)
m

Map(center=[33.9137, -98.4934], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'z…

## Loading GeoJSON from a File

In practice, GeoJSON lives in a file. Use Python's `json` module to load it, then pass the result to `GeoJSON(data=...)`.

In [2]:
import json
from ipyleaflet import Map, GeoJSON

with open("data/wichita_falls.geojson") as f:
    wichita_data = json.load(f)

# wichita_data is now a plain Python dict — same structure as above
print(type(wichita_data))
print(f"{len(wichita_data['features'])} features")

<class 'dict'>
6 features


In [3]:
m = Map(center=WICHITA_FALLS, zoom=12)
layer = GeoJSON(data=wichita_data)
m.add(layer)
m

Map(center=[33.9137, -98.4934], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'z…

One `GeoJSON()` call rendered all six features — points, a line, and a polygon — from the file.

## Inspecting the Features

Before displaying data, it is worth checking what geometry types are present. The `type` key on each feature's `geometry` tells you.

In [4]:
for feature in wichita_data["features"]:
    name = feature["properties"]["name"]
    geom_type = feature["geometry"]["type"]
    print(f"{geom_type:12}  {name}")

Point         Wichita Falls City Hall
Point         Midwestern State University
Point         Lake Wichita
Point         Lucy Park
LineString    MSU to City Hall route
Polygon       Lake Wichita Park


## Multiple Layers on One Map

You can call `m.add()` multiple times to stack layers. Each `GeoJSON` object is its own layer — useful when you want to control them independently later.

In [5]:
from ipyleaflet import Map, GeoJSON

# Split the features by geometry type
points = {"type": "FeatureCollection", "features": [
    f for f in wichita_data["features"] if f["geometry"]["type"] == "Point"
]}
lines = {"type": "FeatureCollection", "features": [
    f for f in wichita_data["features"] if f["geometry"]["type"] == "LineString"
]}
polygons = {"type": "FeatureCollection", "features": [
    f for f in wichita_data["features"] if f["geometry"]["type"] == "Polygon"
]}

m = Map(center=WICHITA_FALLS, zoom=12)
m.add(GeoJSON(data=points))
m.add(GeoJSON(data=lines))
m.add(GeoJSON(data=polygons))
m

Map(center=[33.9137, -98.4934], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'z…

## Removing a Layer

Keep a reference to the layer object and call `m.remove()` to take it off the map.

In [ ]:
from ipyleaflet import Map, GeoJSON

m = Map(center=WICHITA_FALLS, zoom=12)

point_layer = GeoJSON(data=points)
poly_layer  = GeoJSON(data=polygons)

m.add(point_layer)
m.add(poly_layer)

# Remove just the polygon layer — point layer stays
m.remove(poly_layer)

m

Map(center=[33.9137, -98.4934], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'z…

## Exercise A

Load `data/meteorites.geojson` and display all features as a single `GeoJSON` layer on a world-scale map (`center=(20, 0)`, `zoom=2`).

In [1]:
import json
from ipyleaflet import Map, GeoJSON

with open("data/meteorites.geojson") as f:
    meteorites = json.load(f)

m = Map(center=(20, 0), zoom=2)

layer = GeoJSON(data=meteorites)

m.add(layer)

m

Map(center=[20, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text…

## Exercise B

Using `data/wichita_falls.geojson`:

1. Print the count of features for each geometry type
2. Display all features on one map with each geometry type as a **separate named layer**

In [1]:
import json
from ipyleaflet import Map, GeoJSON

with open("data/wichita_falls.geojson") as f:
    features = json.load(f)["features"]

# Count geometry types
counts = {}

for feature in features:
    geom_type = feature["geometry"]["type"]

    if geom_type not in counts:
        counts[geom_type] = 0

    counts[geom_type] += 1

# Print counts
for geom_type, count in counts.items():
    print(f"{geom_type}: {count}")

# Separate features by type
points = {
    "type": "FeatureCollection",
    "features": [f for f in features if f["geometry"]["type"] == "Point"]
}

lines = {
    "type": "FeatureCollection",
    "features": [f for f in features if f["geometry"]["type"] == "LineString"]
}

polygons = {
    "type": "FeatureCollection",
    "features": [f for f in features if f["geometry"]["type"] == "Polygon"]
}

# Create map
m = Map(center=(33.9137, -98.4934), zoom=12)

# Add layers
m.add(GeoJSON(data=points))
m.add(GeoJSON(data=lines))
m.add(GeoJSON(data=polygons))

m

Point: 4
LineString: 1
Polygon: 1


Map(center=[33.9137, -98.4934], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'z…

---

## Check Your Understanding

Load `data/wichita_falls.geojson` and display **only the polygon features** on a map. Print the name of each polygon feature before displaying the map.

```python
# your answer here
```

In [2]:
import json
from ipyleaflet import Map, GeoJSON

with open("data/wichita_falls.geojson") as f:
    data = json.load(f)

# Get only polygon features
polygon_features = [
    f for f in data["features"]
    if f["geometry"]["type"] == "Polygon"
]

# Print polygon names
for feature in polygon_features:
    print(feature["properties"]["name"])

# Create polygon FeatureCollection
polygons = {
    "type": "FeatureCollection",
    "features": polygon_features
}

# Display map
m = Map(center=(33.9137, -98.4934), zoom=12)

polygon_layer = GeoJSON(data=polygons)

m.add(polygon_layer)

m

Lake Wichita Park


Map(center=[33.9137, -98.4934], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'z…

## Next

In [03 — Map Controls](./03-Map_Control.ipynb), we add UI elements — layer toggles, a scale bar, fullscreen, and a minimap — to make the map more usable.